# soccer-cv quickstart

[![GitHub](https://img.shields.io/badge/GitHub-soccer--cv-black?logo=github)](https://github.com/granthohol/soccer-cv)
[![PyPI](https://img.shields.io/pypi/v/soccer-cv.svg)](https://pypi.org/project/soccer-cv/)

This notebook installs [`soccer-cv`](https://github.com/granthohol/soccer-cv) and runs three of its pipelines end-to-end on the sample broadcast clip bundled with the repo — no local setup required. Everything below runs on CPU or GPU; a free Colab GPU runtime (**Runtime → Change runtime type → T4 GPU**) will make it noticeably faster but isn't required for this short clip.

For the full pipeline list, installation options, and how the library works under the hood, see the [README](https://github.com/granthohol/soccer-cv#readme).

## 1. Install

In [ ]:
# Colab already ships a CUDA-matched torch build, so we deliberately don't touch torch here.
!pip install -q "sports@git+https://github.com/roboflow/sports.git@main" "soccer-cv[cuda,data]"

## 2. Get the sample clip

Shallow-clone the repo for its bundled sample video, then trim it to a few seconds so this notebook runs quickly end-to-end. Swap in your own video by uploading it and pointing `CLIP` at it.

In [ ]:
!git clone --depth 1 -q https://github.com/granthohol/soccer-cv.git
%cd soccer-cv

# Trim to a short clip so the demo runs fast; drop -t to process the full video.
!ffmpeg -y -i media/121364_0.mp4 -t 6 -c:v libx264 -c:a aac -loglevel error clip.mp4

CLIP = "clip.mp4"

In [ ]:
import subprocess
from IPython.display import Video, display

def show_video(path, width=480):
    """soccer-cv writes mp4v output; re-encode to H.264 so it plays inline here."""
    h264_path = path.replace(".mp4", "_h264.mp4")
    subprocess.run(
        ["ffmpeg", "-y", "-i", path, "-vcodec", "libx264", "-loglevel", "error", h264_path],
        check=True,
    )
    display(Video(h264_path, embed=True, width=width))

## 3. Team shape — 2D projection with convex-hull team shapes

Projects players onto the canonical 2D pitch and overlays each team's convex hull (area/width/depth), same as the GIF at the top of the README.

In [ ]:
from soccer_cv.pipelines.team_shape import write_team_shape_video

write_team_shape_video(CLIP, "out_team_shape.mp4")
show_video("out_team_shape.mp4")

## 4. Possession

Estimates rolling ball possession per team (nearest player to the ball in pitch space) and renders it as a HUD bar.

In [ ]:
from soccer_cv.pipelines.possession import write_possession_2d_video

pct_team0, pct_team1 = write_possession_2d_video(CLIP, "out_possession.mp4")
print(f"Team 0: {pct_team0:.1%}  |  Team 1: {pct_team1:.1%}")
show_video("out_possession.mp4")

## 5. Tracking + player kinetic stats

Tracks every player with persistent IDs, smooths trajectories with a Kalman filter, and derives per-player distance/speed/acceleration — the same table shown in the README.

In [ ]:
from soccer_cv.pipelines.tracking import write_tracking_video, summarize_player_stats

write_tracking_video(CLIP, "out_tracking.mp4")
show_video("out_tracking.mp4")

# write_tracking_video also drops a per-frame metrics CSV next to the output video
summarize_player_stats("out_tracking_metrics.csv")

## Next steps

- Try `soccer_cv.pipelines.voronoi.write_voronoi_2d_video` (team control Voronoi diagram) or `soccer_cv.pipelines.player_heatmaps` (per-player / per-team heatmaps) the same way.
- Drop the `-t 6` trim above (or point `CLIP` at your own footage) to run on a full clip.
- Full pipeline list, installation options, model training details, and known limitations: see the [README](https://github.com/granthohol/soccer-cv#readme).